In [39]:
import json
import re
from pathlib import Path
from collections import defaultdict
from statistics import mean

root = Path("output_omd_sweep/resnet18/cifar10/forget_10.0%/RL")
rows = []

for p in root.glob("*/*/*/evaluation_result.json"):
    parts = p.parts
    method = parts[-4]
    seed_tag = parts[-3]
    hp_tag = parts[-2]

    m = re.match(r"seed_(\d+)_train_(\d+)", seed_tag)
    h = re.match(r"eta_(.+)_rw_(.+)_fw_(.+)", hp_tag)
    if not m or not h:
        continue

    seed = int(m.group(1))
    train_seed = int(m.group(2))
    eta = h.group(1).replace("p", ".").replace("_neg_", "-")
    rw = h.group(2).replace("p", ".").replace("_neg_", "-")
    fw = h.group(3).replace("p", ".").replace("_neg_", "-")

    data = json.loads(p.read_text())

    if data is None:
        continue
    if not isinstance(data, dict):
        continue

    acc = data.get("accuracy") or data.get("accuracy") or {}
    if not isinstance(acc, dict):
        continue

    retain = acc.get("retain")
    forget = acc.get("forget")
    test = acc.get("test")
    if None in (retain, forget, test):
        continue

    rows.append(
        {
            "method": method,
            "seed": seed,
            "train_seed": train_seed,
            "eta": float(eta),
            "rw": float(rw),
            "fw": float(fw),
            "retain": float(retain),
            "forget": float(forget),
            "test": float(test),
            "score": float(retain) + float(forget) + float(test),
        }
    )

groups = defaultdict(list)
for row in rows:
    key = (row["method"], row["eta"], row["rw"], row["fw"])
    groups[key].append(row)

summary = []
for key, items in groups.items():
    method, eta, rw, fw = key
    summary.append(
        {
            "method": method,
            "eta": eta,
            "rw": rw,
            "fw": fw,
            "n": len(items),
            "retain": mean(x["retain"] for x in items),
            "forget": mean(x["forget"] for x in items),
            "test": mean(x["test"] for x in items),
            "score": mean(x["score"] for x in items),
        }
    )

summary.sort(key=lambda x: x["score"], reverse=True)

by_method = defaultdict(list)
for row in summary:
    by_method[row["method"]].append(row)

for method in sorted(by_method):
    print(f"\nTop combinations for {method}")
    for row in by_method[method][:10]:
        print(
            f'{row["method"]:12s} eta={row["eta"]:<5g} rw={row["rw"]:<4g} fw={row["fw"]:<4g} '
            f'n={row["n"]} retain={row["retain"]:.2f} forget={row["forget"]:.2f} '
            f'test={row["test"]:.2f} score={row["score"]:.2f}'
        )



Top combinations for ada_omd_tch_eg
ada_omd_tch_eg eta=0.01  rw=1    fw=1    n=4 retain=99.80 forget=47.10 test=94.99 score=241.88
ada_omd_tch_eg eta=0.03  rw=1    fw=1    n=4 retain=99.81 forget=33.06 test=94.95 score=227.82
ada_omd_tch_eg eta=0.1   rw=1    fw=1    n=4 retain=99.81 forget=2.42 test=94.90 score=197.12
ada_omd_tch_eg eta=0.3   rw=1    fw=1    n=4 retain=99.81 forget=2.17 test=94.89 score=196.87

Top combinations for omd_tch_eg
omd_tch_eg   eta=0.01  rw=1    fw=1    n=5 retain=99.68 forget=93.60 test=94.93 score=288.21
omd_tch_eg   eta=0.005 rw=1    fw=1    n=5 retain=99.70 forget=93.36 test=94.96 score=288.03
omd_tch_eg   eta=0.05  rw=1    fw=1    n=5 retain=99.59 forget=93.50 test=94.80 score=287.89
omd_tch_eg   eta=0.02  rw=1    fw=1    n=5 retain=99.63 forget=93.20 test=94.86 score=287.70
omd_tch_eg   eta=0.03  rw=1    fw=1    n=5 retain=99.61 forget=93.14 test=94.82 score=287.56
omd_tch_eg   eta=0.08  rw=1    fw=1    n=5 retain=99.58 forget=93.09 test=94.78 score=2

In [31]:
from pathlib import Path
import itertools

root = Path("output_omd_sweep/resnet18/cifar10/forget_10.0%/RL")

methods = ["omd_tch_eg", "omd_tch_pgd"]
seeds = [1, 2, 3]
etas_eg = ["0p005", "0p01", "0p02", "0p03", "0p05", "0p08", "0p1", "0p15", "0p2", "0p3"]
etas_pgd = ["0p005", "0p01", "0p02", "0p03", "0p05"]
rw = "1p0"
fw = "1p0"

for method in methods:
    etas = etas_eg if method == "omd_tch_eg" else etas_pgd
    print(f"\nMissing for {method}:")
    for seed, eta in itertools.product(seeds, etas):
        p = root / method / f"seed_{seed}_train_1" / f"eta_{eta}_rw_{rw}_fw_{fw}" / "evaluation_result.json"
        if not p.exists():
            print(p)



Missing for omd_tch_eg:

Missing for omd_tch_pgd:


In [37]:
from pathlib import Path
import json
import re

root = Path("output_omd_sweep/resnet18/cifar10/forget_10.0%/RL")

rows = []
skipped = []

flat_pat = re.compile(r"seed_(\d+)_train_(\d+)_eta_(.+)_rw_(.+)_fw_(.+)")
seed_pat = re.compile(r"seed_(\d+)_train_(\d+)")
hp_pat = re.compile(r"eta_(.+)_rw_(.+)_fw_(.+)")

for p in root.rglob("evaluation_result.json"):
    rel = p.relative_to(root)
    rel_parts = rel.parts

    method = None
    seed = None
    train_seed = None
    eta = None
    rw = None
    fw = None

    if len(rel_parts) == 3:
        method, flat_tag, _ = rel_parts
        m = flat_pat.fullmatch(flat_tag)
        if not m:
            skipped.append(("flat_path_parse", str(p)))
            continue
        seed = int(m.group(1))
        train_seed = int(m.group(2))
        eta = m.group(3)
        rw = m.group(4)
        fw = m.group(5)

    elif len(rel_parts) == 4:
        method, seed_tag, hp_tag, _ = rel_parts

        m = seed_pat.fullmatch(seed_tag)
        h = hp_pat.fullmatch(hp_tag)
        if not m or not h:
            skipped.append(("nested_path_parse", str(p)))
            continue

        seed = int(m.group(1))
        train_seed = int(m.group(2))
        eta = h.group(1)
        rw = h.group(2)
        fw = h.group(3)

    else:
        skipped.append(("unexpected_depth", str(p)))
        continue

    try:
        data = json.loads(p.read_text())
    except Exception as e:
        skipped.append((f"json_parse_error:{type(e).__name__}", str(p)))
        continue

    if data is None or not isinstance(data, dict):
        skipped.append(("bad_json", str(p)))
        continue

    acc = data.get("accuracy") or {}
    if not isinstance(acc, dict):
        skipped.append(("bad_accuracy", str(p)))
        continue

    retain = acc.get("retain")
    forget = acc.get("forget")
    test = acc.get("test")
    if None in (retain, forget, test):
        skipped.append(("missing_metric", str(p)))
        continue

    rows.append(
        {
            "method": method,
            "seed": seed,
            "train_seed": train_seed,
            "eta": eta,
            "rw": rw,
            "fw": fw,
            "path": str(p),
        }
    )

print("ROWS")
for row in sorted(rows, key=lambda x: (x["method"], x["eta"], x["seed"])):
    print(row["method"], row["eta"], row["seed"], row["path"])

print("\nSKIPPED")
for reason, path in skipped:
    print(reason, path)


ROWS
ada_omd_tch_eg 0p01 1 output_omd_sweep/resnet18/cifar10/forget_10.0%/RL/ada_omd_tch_eg/seed_1_train_1/eta_0p01_rw_1p0_fw_1p0/evaluation_result.json
ada_omd_tch_eg 0p01 2 output_omd_sweep/resnet18/cifar10/forget_10.0%/RL/ada_omd_tch_eg/seed_2_train_1/eta_0p01_rw_1p0_fw_1p0/evaluation_result.json
ada_omd_tch_eg 0p01 3 output_omd_sweep/resnet18/cifar10/forget_10.0%/RL/ada_omd_tch_eg/seed_3_train_1/eta_0p01_rw_1p0_fw_1p0/evaluation_result.json
ada_omd_tch_eg 0p01 4 output_omd_sweep/resnet18/cifar10/forget_10.0%/RL/ada_omd_tch_eg/seed_4_train_1/eta_0p01_rw_1p0_fw_1p0/evaluation_result.json
ada_omd_tch_eg 0p03 1 output_omd_sweep/resnet18/cifar10/forget_10.0%/RL/ada_omd_tch_eg/seed_1_train_1/eta_0p03_rw_1p0_fw_1p0/evaluation_result.json
ada_omd_tch_eg 0p03 2 output_omd_sweep/resnet18/cifar10/forget_10.0%/RL/ada_omd_tch_eg/seed_2_train_1/eta_0p03_rw_1p0_fw_1p0/evaluation_result.json
ada_omd_tch_eg 0p03 3 output_omd_sweep/resnet18/cifar10/forget_10.0%/RL/ada_omd_tch_eg/seed_3_train_1/eta_0

In [33]:
from pathlib import Path
import itertools

root = Path("output_omd_sweep/resnet18/cifar10/forget_10.0%/RL")

seeds = [1, 2, 3, 4, 5]
eg_etas = ["0p005", "0p01", "0p02", "0p03", "0p05", "0p08", "0p1", "0p15", "0p2", "0p3"]
pgd_etas = ["0p005", "0p01", "0p02", "0p03", "0p05"]
rw = "1p0"
fw = "1p0"

for method, etas in {
    "omd_tch_eg": eg_etas,
    "omd_tch_pgd": pgd_etas,
}.items():
    print(f"\nMissing for {method}:")
    for seed, eta in itertools.product(seeds, etas):
        p = root / method / f"seed_{seed}_train_1" / f"eta_{eta}_rw_{rw}_fw_{fw}" / "evaluation_result.json"
        if not p.exists():
            print(p)



Missing for omd_tch_eg:

Missing for omd_tch_pgd:
